# 1. Alpha-Beta Pruning

Aplică Alpha-Beta Pruning pe jocul X&0

In [5]:
class Nod:
    def __init__(self, info, parent=None, euristica=0):
        self.info = info
        self.parent = parent
        self.euristica = euristica

    def __str__(self):
        out = ""
        for linie in self.info:
            out += "|".join(linie) + "\n"
            out += "-----\n"
        out += f"Scor Euristic: {self.euristica}\n"
        return out

class Game:
    MAX = 'X'
    MIN = '0'
    GOL = '.'

    def __init__(self, tabla_start=None):
        if tabla_start is None:
            self.nodStart = Nod(((self.GOL, self.GOL, self.GOL),
                                 (self.GOL, self.GOL, self.GOL),
                                 (self.GOL, self.GOL, self.GOL)))
        else:
            self.nodStart = Nod(tabla_start)

    def jucator_opus(self, jucator):
        return self.MIN if jucator == self.MAX else self.MAX

    def succesori(self, nod, jucator):
        suc = []
        tabla = nod.info
        for i in range(3):
            for j in range(3):
                if tabla[i][j] == self.GOL:
                    tabla_noua = [list(rand) for rand in tabla]
                    tabla_noua[i][j] = jucator
                    stare_noua = tuple(tuple(rand) for rand in tabla_noua)
                    suc.append(Nod(stare_noua, parent=nod))
        return suc

    def scop(self, tabla):
        linii = []
        for i in range(3):
            linii.append(tabla[i])
            linii.append((tabla[0][i], tabla[1][i], tabla[2][i]))
        linii.append((tabla[0][0], tabla[1][1], tabla[2][2]))
        linii.append((tabla[0][2], tabla[1][1], tabla[2][0]))

        for linie in linii:
            if linie.count(self.MAX) == 3: return self.MAX
            elif linie.count(self.MIN) == 3: return self.MIN

        for linie in tabla:
            if self.GOL in linie: return False
        return "remiza"

    def calculeaza_euristica(self, tabla):
        linii = []
        for i in range(3):
            linii.append(tabla[i]) 
            linii.append((tabla[0][i], tabla[1][i], tabla[2][i])) 
            
        linii.append((tabla[0][0], tabla[1][1], tabla[2][2])) 
        linii.append((tabla[0][2], tabla[1][1], tabla[2][0])) 

        scor_total = 0
        for linie in linii:
            x_count = linie.count(self.MAX)
            o_count = linie.count(self.MIN) 

            if x_count > 0 and o_count > 0:
                continue 
            if x_count == 1: scor_total += 1
            elif x_count == 2: scor_total += 10
            elif x_count == 3: scor_total += 100
                
            if o_count == 1: scor_total -= 1
            elif o_count == 2: scor_total -= 10
            elif o_count == 3: scor_total -= 100

        return scor_total

def minmax(nod, jucator, adancime, game, alpha, beta):
    if adancime == 0 or game.scop(nod.info) != False:
        nod.euristica = game.calculeaza_euristica(nod.info)
        return nod

    succesori = game.succesori(nod, jucator)
    
    if jucator == game.MAX:
        max_val = -float('inf')
        best_node = None
        for suc in succesori:
            rezultat = minmax(suc, game.jucator_opus(jucator), adancime - 1, game, alpha, beta)
            if rezultat.euristica > max_val:
                max_val = rezultat.euristica
                best_node = suc

            alpha = max(alpha,  max_val)
            if beta <= alpha:
                break                
                
        best_node.euristica = max_val
        nod.euristica = max_val
        return best_node
        
    else:
        min_val = float('inf')
        best_node = None
        for suc in succesori:
            rezultat = minmax(suc, game.jucator_opus(jucator), adancime - 1, game, alpha, beta)
            if rezultat.euristica < min_val:
                min_val = rezultat.euristica
                best_node = suc
            
            beta = min(beta,  min_val)
            if beta <= alpha:
                break 
                
        best_node.euristica = min_val
        nod.euristica = min_val
        return best_node

joc = Game()

print("Tabla de start:")
print(joc.nodStart)


output = minmax(joc.nodStart, joc.MAX, 3, joc, -float('inf'), float('inf'))

print(output)

Tabla de start:
.|.|.
-----
.|.|.
-----
.|.|.
-----
Scor Euristic: 0

.|.|.
-----
.|X|.
-----
.|.|.
-----
Scor Euristic: 12



# 2. Blackjack

Implementati jocul Blackjack folosind algoritmul expectimax. Nu uitati ca la blackjack dealer-ul mereu da hit la 16 sau mai putin, si mereu stand daca are cel putin 17, deci nu aveti minimizer. Este, practic, purely player vs nature.

Se joaca cu "stand all 17s", i.e. daca dealer-ul are 17(contine un as care este evaluat la 11), trebuie sa dea stand(favorizeaza jucatorul putin).

upcard e cartea dealer-ului pe care o vedeti.

In [ ]:
#ramane de implementat alg
#puteti hardcoda probabilitati sau cu random sampling(sau ce alte idei aveti voi). astea mi se par mie cele mai usoare de implementat.
class Blackjack:
    def __init__(self, carti, upcard):
        self.carti_player = carti
        self.upcard_dealer = upcard
        self.probabilitati = {
            2: 1/13, 3: 1/13, 4: 1/13, 5: 1/13, 6: 1/13,
            7: 1/13, 8: 1/13, 9: 1/13, 10: 1/13,
            'J': 1/13, 'Q': 1/13, 'K': 1/13, 'A': 1/13
        }
        #carti e lista cu cartile jucatorului. n-as pune direct suma ca sa fie mai usor sa tratati cazul cu as
    def calcul_scor(self, carti):
        scor = 0
        asi = 0
        for carte in carti:
            if carte in ['J', 'K', 'Q']:
                scor += 10
            elif carte == 'A':
                asi += 1
                scor += 11
            else:
                scor += int(carte)

        while scor > 21 and asi > 0:
            scor -= 10
            asi -= 1

        return scor

    def eval_stand(self, scor_player, carti_dealer):

        scor_dealer = self.calcul_scor(carti_dealer)
        if scor_dealer > 21:
            return 1
        if scor_dealer >= 17:
            if scor_player > scor_dealer:
                return 1
            elif scor_dealer > scor_player:
                return -1
            else:
                return 0
            
        valoare_asteptata = 0
        for carte, prob in self.probabilitati.items():
            valoare_asteptata += prob * self.eval_stand(scor_player, carti_dealer + [carte])
            
        return valoare_asteptata
            


    def expectimax(self, adancime, carti_player): #trebuie adancime > 1 altfel o sa cam favorizeze stand oricand, mai ales cu probabilitati hardcodate. be careful
        #hit-ul vrem sa il evaluam recursiv.
        
        scor_player = self.calcul_scor(carti_player)

        if scor_player > 21:
            return -1
    
        valoare_stand = self.eval_stand(scor_player, [self.upcard_dealer])

        if adancime == 0:
            return valoare_stand 
        
        valoare_hit = 0
        for carte, prob in self.probabilitati.items():
            val_viitoare = self.expectimax(adancime - 1, carti_player + [carte])
            valoare_hit += prob * val_viitoare

        if valoare_hit > valoare_stand:
            return valoare_hit
        else:
            return valoare_stand


    def play(self):
        #intai player face ce face, dupa dealer si afisam castigatorul.
        scor = self.calcul_scor(self.carti_player)
        adancime = 3
        valoare_estimata= self.expectimax(adancime, self.carti_player)
        
        print(f"-> Utilitate estimat: {valoare_estimata:.2f} ")


joc0 = Blackjack(carti=[9, 7], upcard=10)
joc0.play()

joc1 = Blackjack(carti=['K', 'Q'], upcard=6)
joc1.play()

joc2 = Blackjack(carti=[8, 2], upcard=10)
joc2.play()

joc3 = Blackjack(carti=[10, 3], upcard=5)
joc3.play()

joc4 = Blackjack(carti=['A', 7], upcard=9)
joc4.play()

-> Utilitate estimat: -0.57 
-> Utilitate estimat: 0.70 
-> Utilitate estimat: -0.05 
-> Utilitate estimat: -0.17 
-> Utilitate estimat: -0.10 
